# 02 - Storm-track splitting

Thin caller over `nfip.hurdat`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))   # the nfip package
sys.path.insert(0, os.path.abspath('src'))       # existing ST_Cluster.py, if used
import numpy as np, pandas as pd
from nfip import data, config, hurdat

## Parameters

In [ ]:
optimal_cluster = 'st_cluster_final_gid'
R_KM     = 50.0
T_HOURS  = 72.0
EXCLUDE_CLUSTER = -1
POINT_SIZE = 10

## Basemaps + clusters

In [ ]:
gdf_counties = data.try_load_counties(crs=config.CRS_EQD)
gdf_states   = data.try_load_states(crs=config.CRS_EQD)

In [ ]:
clustered_claims = pd.read_csv(config.PATHS.clusters_new_csv)
clustered_claims['countyCode'] = clustered_claims['countyCode'].astype(int).astype(str).str.zfill(5)
clustered_claims['dateOfLoss'] = pd.to_datetime(clustered_claims['dateOfLoss']).dt.tz_localize(None)

claims = (clustered_claims[[optimal_cluster, 'dateOfLoss', 'latitude', 'longitude']]
          .dropna(subset=[optimal_cluster, 'dateOfLoss', 'latitude', 'longitude'])
          .rename(columns={optimal_cluster: 'cluster_id', 'latitude': 'claim_lat', 'longitude': 'claim_lon'})
          .copy())
claims['date_floor'] = claims['dateOfLoss'].dt.floor('D')

## HURDAT2 tracks + space-time matching

In [ ]:
hurdat_df = hurdat.load_hurdat()
tracks = hurdat.build_tracks_table(hurdat_df)

In [ ]:
matches = hurdat.match_claims_to_tracks(claims, tracks, r_km=R_KM, t_hours=T_HOURS)
print('match rows:', len(matches))

In [ ]:
tracks_intersecting = matches[['storm_id']].drop_duplicates()
per_year_tracks = (matches[['year','storm_id']].drop_duplicates()
                   .groupby('year')['storm_id'].nunique().sort_index())
print(f'Total tracks intersecting clusters (R<={R_KM} km, |dt|<={T_HOURS} h): {len(tracks_intersecting)}')
print(per_year_tracks.to_string())

## Multi-track clusters + SVM split

In [ ]:
valid_matches, multi_track_clusters = hurdat.find_multi_track_clusters(matches, exclude_cluster=EXCLUDE_CLUSTER)
print(f'Clusters to split (>=2 tracks, excluding {EXCLUDE_CLUSTER}): {len(multi_track_clusters)}')

In [ ]:
hurdat_ll, gdf_tracks = hurdat.build_track_lines(hurdat_df)

In [ ]:
assignments = hurdat.split_multitrack_clusters(
    clustered_claims, optimal_cluster, valid_matches, multi_track_clusters,
    hurdat_ll, gdf_tracks, gdf_counties=gdf_counties, gdf_states=gdf_states,
    point_size=POINT_SIZE)

## Write split assignments

In [ ]:
if assignments:
    assign_all = pd.concat(assignments, ignore_index=True)
    id_to_storm = assign_all.drop_duplicates('id').set_index('id')['svm_storm_id']
    id_to_conf  = assign_all.drop_duplicates('id').set_index('id')['svm_confidence']
    mt_mask = clustered_claims[optimal_cluster].isin(multi_track_clusters)
    clustered_claims.loc[mt_mask, 'svm_storm_id'] = clustered_claims.loc[mt_mask, 'id'].map(id_to_storm)
    clustered_claims.loc[mt_mask, 'svm_confidence'] = clustered_claims.loc[mt_mask, 'id'].map(id_to_conf)
    assigned = clustered_claims.loc[mt_mask & clustered_claims['svm_storm_id'].notna()]
    print(f'SVM assignments written for {len(assigned):,} rows across {assigned[optimal_cluster].nunique()} multi-track clusters.')
else:
    print('No assignments produced.')

In [ ]:
if 'temporal_cluster_gid' not in clustered_claims.columns:
    for cand in ['temporal_cluster', 'temporal_cluster_id']:
        if cand in clustered_claims.columns:
            clustered_claims['temporal_cluster_gid'] = clustered_claims[cand]; break
    clustered_claims.setdefault('temporal_cluster_gid', pd.NA)

has_assign = clustered_claims['svm_storm_id'].notna()
clustered_claims.loc[has_assign, 'split_cluster_id'] = (
    clustered_claims.loc[has_assign, 'st_cluster_final_gid'].astype(str) + '_' +
    clustered_claims.loc[has_assign, 'svm_storm_id'].astype(str))
cols_out = ['id','dateOfLoss','longitude','latitude','temporal_cluster_gid','st_cluster_final_gid','split_cluster_id']
subset_df = (clustered_claims.loc[has_assign, cols_out]
             .sort_values(['st_cluster_final_gid','split_cluster_id','dateOfLoss']).copy())
subset_df.to_csv('hurr_cluster_split_subset1.csv', index=False)
print(f'Exported {len(subset_df):,} rows')